### Question 1. Instrument the agent with Logfire

In [1]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-us.pydantic.dev/yonix500/starter-project

In [2]:
from dotenv import load_dotenv

load_dotenv()

from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

In [4]:
# Download the FAQ and build the search index

documents = load_faq_data()
index = build_index(documents)

# Inject the index into the agent via the dependency container
deps = SearchDeps(index=index)


In [22]:

# Ask a question. run_sync blocks until the agent is done;
# the agent may call search multiple times before answering.
question = 'How do I run Ollama locally??'
result = await faq_agent.run(question, deps=deps)

print(result.output)

17:11:42.498 faq_agent run
17:11:42.517   chat gpt-5.4-mini
17:11:44.571   running tool: search
17:11:44.598   chat gpt-5.4-mini
Yes — for Ollama locally, the course FAQ says:

1. **Install Ollama** from: https://ollama.com/download  
   - **macOS**: download the `.pkg`
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Run a model locally**:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like prompt.

3. **Test the local server**:
   ```bash
   curl http://localhost:11434
   ```

4. **Optional Python client**:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```

If you want, I can also help with the course-specific Ollama setup or troubleshooting. A

### Question 2. Load traces into DuckDB with dlt

In [ ]:
import os
import dlt
from dlt.sources.rest_api import RESTAPIConfig, rest_api_resources


@dlt.source
def logfire_source(
    access_token: str = os.getenv("LOGFIRE_READ_TOKEN")
):
    if not access_token:
        raise ValueError("LOGFIRE_READ_TOKEN is not set")

    config: RESTAPIConfig = {
        "client": {
            "base_url": "https://logfire-eu.pydantic.dev/v1/",
            "auth": {
                "type": "bearer",
                "token": access_token,
            },
        },
        "resources": [
            {
                "name": "query",
                "endpoint": {
                    "path": "query",
                    "method": "GET",
                },
            }
        ],
    }

    yield from rest_api_resources(config)


def get_data() -> None:
    pipeline = dlt.pipeline(
        pipeline_name="logfire_pipeline",
        destination="duckdb",
        dataset_name="logfire_data",
    )

    load_info = pipeline.run(logfire_source())
    print(load_info)


if __name__ == "__main__":
    get_data()

### Question 3. Query traces with an agent

In [37]:
import json

total_input_tokens = 0

# collect traces into a list (support single `trace`, `traces`, or fetch from DuckDB)
if "trace" in globals():
    traces_list = [trace]
elif "traces" in globals():
    traces_list = traces
else:
    traces_list = []
    if "con" in globals():
        try:
            rows = con.execute("SELECT trace FROM agent_traces").fetchall()
            for row in rows:
                traces_list.append(row[0])
        except Exception:
            pass

def extract_spans(item):
    if hasattr(item, "spans"):
        return item.spans
    if isinstance(item, dict):
        return item.get("spans", [])
    if isinstance(item, (str, bytes)):
        try:
            obj = json.loads(item)
            return obj.get("spans", [])
        except Exception:
            return []
    return []

for tr in traces_list:
    for span in extract_spans(tr):
        # span.attributes may be an object or dict
        attrs = getattr(span, "attributes", None)
        if attrs is None and isinstance(span, dict):
            attrs = span.get("attributes", {})
        if isinstance(attrs, str):
            try:
                attrs = json.loads(attrs)
            except Exception:
                attrs = {}
        input_tokens = None
        if isinstance(attrs, dict):
            input_tokens = attrs.get("gen_ai.usage.input_tokens")
        elif hasattr(attrs, "get"):
            input_tokens = attrs.get("gen_ai.usage.input_tokens")
        if input_tokens is not None:
            try:
                total_input_tokens += int(input_tokens)
            except Exception:
                try:
                    total_input_tokens += int(float(input_tokens))
                except Exception:
                    pass

print("Total input tokens:", total_input_tokens)

Total input tokens: 0
